# FormFlow – Survey Response Prediction
## Decision Tree Classifier
Predicts whether survey feedback is Positive, Neutral, or Negative.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries loaded.')

## Step 1: Generate / Load Dataset

In [ ]:
import os
if os.path.exists('survey_dataset.csv'):
    df = pd.read_csv('survey_dataset.csv')
    print('Loaded existing dataset.')
else:
    np.random.seed(42)
    n = 300
    form_types = np.random.choice(['feedback','event','quiz','registration','research'], n)
    num_questions = np.random.randint(3, 15, n)
    num_responses = np.random.randint(1, 100, n)
    avg_ans_length = np.random.randint(5, 150, n)
    submission_hour = np.random.randint(0, 24, n)
    labels = []
    for i in range(n):
        score = 0
        if avg_ans_length[i] > 60: score += 1
        if num_responses[i] > 30: score += 1
        if num_questions[i] < 6: score += 1
        if submission_hour[i] in range(9, 21): score += 1
        if avg_ans_length[i] < 15: score -= 1
        score += np.random.randint(-1, 2)
        labels.append(2 if score >= 3 else (1 if score >= 1 else 0))
    le = LabelEncoder()
    df = pd.DataFrame({
        'form_type': le.fit_transform(form_types),
        'num_questions': num_questions,
        'num_responses': num_responses,
        'avg_ans_length': avg_ans_length,
        'submission_hour': submission_hour,
        'label': labels
    })
    df.to_csv('survey_dataset.csv', index=False)
    print('Generated and saved dataset.')

print(df.shape)
df.head()

## Step 2: Data Overview

In [ ]:
print('Label distribution:')
print(df['label'].value_counts().rename({0:'Negative', 1:'Neutral', 2:'Positive'}))
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().rename({0:'Negative',1:'Neutral',2:'Positive'}).plot(kind='bar', ax=axes[0], color=['#e76f51','#f4a261','#52b788'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Count')
df[['num_questions','num_responses','avg_ans_length']].hist(ax=axes[1], bins=15)
plt.tight_layout(); plt.show()

## Step 3: Train/Test Split

In [ ]:
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Step 4: Train Decision Tree

In [ ]:
clf = DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)
print('Model trained.')
print(export_text(clf, feature_names=list(X.columns)))

## Step 5: Evaluate

In [ ]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')
print(classification_report(y_test, y_pred, target_names=['Negative','Neutral','Positive']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Neg','Neu','Pos'], yticklabels=['Neg','Neu','Pos'], cmap='Greens')
plt.title('Confusion Matrix'); plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.plot(kind='bar', color='#2d6a4f', figsize=(7,4))
plt.title('Feature Importances'); plt.tight_layout(); plt.show()

## Step 6: Save Model

In [ ]:
joblib.dump(clf, 'model.pkl')
print('Model saved to model.pkl')

## Step 7: Test with a sample

In [ ]:
model = joblib.load('model.pkl')
sample = pd.DataFrame([{'form_type': 1, 'num_questions': 5, 'num_responses': 40, 'avg_ans_length': 75, 'submission_hour': 14}])
pred = model.predict(sample)[0]
labels = {0: 'Negative Feedback', 1: 'Neutral Feedback', 2: 'Positive Feedback'}
print('Predicted:', labels[pred])